# Visuals (1A+1B)

In [ ]:
source("/maps/projects/perslab/people/lhv464/glp1r_leptin/R/basic_processing.R")
source("/maps/projects/perslab/people/lhv464/glp1r_leptin/R/analysis_wrappers.R")
source("/maps/projects/perslab/people/lhv464/glp1r_leptin/R/helper_functions.R")

library(org.Hs.eg.db)
library(miloR)
library(SingleCellExperiment)
library(tidyverse)
library(DESeq2)
library(Seurat)
library(ggrastr)
library(ggpubr)

targets::tar_load("glp1rleprko_neurons", store = "/maps/projects/perslab/people/lhv464/glp1r_leptin/_targets/")
targets::tar_load("arc_diet_labeled", store = "/maps/projects/perslab/people/lhv464/glp1r_leptin/_targets/")
targets::tar_load("arc_lepip", store = "/maps/projects/perslab/people/lhv464/glp1r_leptin/_targets/")

arc_lepip <- process_seurat(arc_lepip, method = "integrate", batch = "orig.ident", res = 30, dims = 30)  %>% 
    project_umap(query = ., ref = arc_diet_labeled, dims = 30, label_to_transfer="seurat_clusters", 
                 reference.assay = "integrated", query.assay = "integrated") 
arc_lepip$predicted.celltype <- str_replace_all(arc_lepip$predicted.celltype, c("32" = "Irx3", "28" = "Irx3", "21" = "Irx3"))  # Just for Irx3

arc_lepip

In [ ]:
DefaultAssay(arc_lepip) <- "RNA"
lepr_exp <- AverageExpression(arc_lepip, features = "Lepr", group.by = "predicted.celltype")
lepclus  <- names(as.data.frame(lepr_exp$RNA))[which(lepr_exp$RNA > 0.75)]
lepclus
#all_lepr_labels <- c("Agrp" = "AgRP", "12" = "Lepr/Glp1r", "29" = "POMC", "26" = "Tbx19", "28" = "Irx3")

In [ ]:
arc_lepip$predicted.celltype <- ifelse(arc_lepip$predicted.celltype == 12, "Lepr/Glp1r", arc_lepip$predicted.celltype)
arc_lepip$predicted.celltype <- ifelse(arc_lepip$predicted.celltype %in% c(18,29), "Lepr/POMC", arc_lepip$predicted.celltype)
arc_lepip$predicted.celltype <- ifelse(arc_lepip$predicted.celltype %in% c("Agrp",16,43), "Lepr/AgRP", arc_lepip$predicted.celltype)
arc_lepip$predicted.celltype <- ifelse(arc_lepip$predicted.celltype == "Irx3", "Lepr/Irx3", arc_lepip$predicted.celltype)
arc_lepip$predicted.celltype <- ifelse(arc_lepip$predicted.celltype == 26, "Lepr/Tbx19", arc_lepip$predicted.celltype)

In [ ]:
arc_lepip$lepr.plot <- ifelse(arc_lepip$predicted.celltype %in% 
                              c("Lepr/Glp1r", "Lepr/POMC", "Lepr/AgRP", "Lepr/Irx3", "Lepr/Tbx19"), 
                              arc_lepip$predicted.celltype, 
                              NA)

In [ ]:
umap_embed <- as.data.frame(arc_lepip@reductions$umap@cell.embeddings) %>%
  mutate(
    celltype = arc_lepip$lepr.plot)

label <- umap_embed %>%
  group_by(celltype) %>%
  summarize(x = median(umap_1), y = median(umap_2))

In [ ]:
p <- ggplot(umap_embed, aes(x = umap_1, y = umap_2, colour = celltype)) +
  geom_point_rast(size = 0.1, alpha = 0.5) +
  theme_pubr(base_size = 12, base_family = "sans") +
  theme(
    axis.line = element_line(colour = "black", size = 0.4),
    panel.grid = element_blank(),
    panel.border = element_blank(),
    panel.background = element_blank(),
    legend.position = "none",
    axis.title = element_text(face = "bold"),
    axis.text = element_text(face = "bold")
  ) +
  labs(x = "UMAP 1", y = "UMAP 2") +
  geom_text(
    data = label, aes(label = celltype, x = x, y = y),
    size = 6 / .pt,
    fontface = "bold",
    inherit.aes = FALSE
  )

p

In [ ]:
options(repr.plot.height = 5, repr.plot.width = 10)
Idents(arc_lepip) <- arc_lepip$lepr.plot
VlnPlot(arc_lepip, features = "Lepr", pt.size = 0) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) + NoLegend()

# Magma-Milo (1C)

In [ ]:
source("/maps/projects/perslab/people/lhv464/glp1r_leptin/R/basic_processing.R")
source("/maps/projects/perslab/people/lhv464/glp1r_leptin/R/analysis_wrappers.R")
source("/maps/projects/perslab/people/lhv464/glp1r_leptin/R/helper_functions.R")

library(org.Hs.eg.db)
library(miloR)
library(SingleCellExperiment)
library(tidyverse)
library(DESeq2)
library(Seurat)

targets::tar_load("arc_diet_labeled", store = "/maps/projects/perslab/people/lhv464/glp1r_leptin/_targets/")
targets::tar_load("arc_lepip", store = "/maps/projects/perslab/people/lhv464/glp1r_leptin/_targets/")

arc_lepip <- process_seurat(arc_lepip, method = "integrate", batch = "orig.ident", res = 30, dims = 30)  %>% 
    project_umap(query = ., ref = arc_diet_labeled, dims = 30, label_to_transfer="seurat_clusters", 
                 reference.assay = "integrated", query.assay = "integrated") 
arc_lepip$predicted.celltype <- str_replace_all(arc_lepip$predicted.celltype, c("32" = "Irx3", "28" = "Irx3", "21" = "Irx3"))  # Just for Irx3

arc_lepip_backup <- arc_lepip

In [ ]:
arc_lepip <- arc_lepip_backup
arc_lepip <- subset(arc_lepip, time == 24) # Rotating between time points here, then rerunning script

milo_lepip <- seurat_to_milo(arc_lepip, pca_reduction_name = "pca", umap_reduction_name = "umap")

In [ ]:
design <- data.frame(colData(milo_lepip))[,c("ObsID", "group", "orig.ident")]
design <- distinct(design)
rownames(design) <- design$ObsID
## Reorder rownames to match columns of nhoodCounts(milo)
design <- design[colnames(nhoodCounts(milo_lepip)), , drop=FALSE]
contrastHFD <- c("(groupmut_Lep - groupmut_Sal) - (groupwt_Lep - groupwt_Sal)") 
contrastFast <- c("(groupwt_Lep) - (groupwt_Sal)") 

# we need to use the ~ 0 + Variable expression here so that we have all of the levels of our variable as separate columns in our model matrix
da_results_hfd <- testNhoods(milo_lepip, design = ~ 0 + group, design.df = design, model.contrasts = contrastHFD,
                         fdr.weighting="graph-overlap", norm.method="TMM")
da_results_fast <- testNhoods(milo_lepip, design = ~ 0 + group , design.df = design, model.contrasts = contrastFast,
                         fdr.weighting="graph-overlap", norm.method="TMM")

da_results_hfd <- annotateNhoods(milo_lepip, da_results_hfd,  coldata_col = "predicted.celltype")
da_results_hfd <- annotateNhoods(milo_lepip, da_results_hfd,  coldata_col = "group")

da_results_fast <- annotateNhoods(milo_lepip, da_results_fast,  coldata_col = "predicted.celltype")
da_results_fast <- annotateNhoods(milo_lepip, da_results_fast,  coldata_col = "group")
da_results_fast %>% arrange(SpatialFDR)  %>%  head

In [ ]:
# create neighborhood pseudobulk
nh_matrix <- nhoods(milo_lepip)
colnames(nh_matrix) <- 1:ncol(nh_matrix)

# create gene expression by neighborhood matrix
signature_scores <- arc_lepip@assays$RNA@counts
score_sums <- t(Matrix::t(nh_matrix) %*% Matrix::t(signature_scores))

dds <- DESeqDataSetFromMatrix(score_sums, 
                              colData = da_results_hfd, 
                              design = ~predicted.celltype+group)
                              
keep <- rowSums(counts(dds) >= 5) >= 20
table(keep)
dds_filtered <- dds[keep, ]
dds_filtered
vsd <- vst(dds_filtered, blind=FALSE)

In [ ]:
# prep data for magma

spm_scores <- assay(vsd)[!grepl("Gm|mt-|Rik", rownames(assay(vsd))),]

genes_converted <- data.frame(convert_mouse_to_human(rownames(spm_scores))) %>% filter(!duplicated(X1))

genes_entrez <- AnnotationDbi::select(org.Hs.eg.db,
                            keys = unique(genes_converted$X2),
                            columns = c("ENTREZID","ENSEMBL", "SYMBOL"),
                            keytype = "SYMBOL")  %>% 
                            filter(!duplicated(SYMBOL))

humdeg <- inner_join(genes_entrez, genes_converted, by=c("SYMBOL" = "X2"))  %>% 
    inner_join(spm_scores  %>% data.frame()  %>% janitor::clean_names()  %>%  rownames_to_column("mouse_gene"), by=c("X1"="mouse_gene"))  %>% dplyr::select(c(-SYMBOL,-X1,-ENSEMBL))  %>%
    dplyr::rename(GENE=ENTREZID)  %>% 
    filter(!duplicated(GENE), !is.na(GENE)) 

humdeg$Average <- rowMeans(humdeg[,-1])

write.table(humdeg, file="/projects/perslab/people/jmg776/ad_hoc/dylan_MAGMA-milo/milo_lepinject.txt", sep = "\t", quote=F, row.names=F)

In [ ]:
# Run this in a terminal after initial magma run. 

magma --gene-results /projects/perslab/apps/MAGMA/out/SystolicBP_GCST90310294/SystolicBP_GCST90310294.genes.raw \
      --gene-covar /projects/perslab/people/jmg776/ad_hoc/dylan_MAGMA-milo/milo_lepinject.txt \
      --model direction=greater condition-hide=Average --out /projects/perslab/people/jmg776/ad_hoc/dylan_MAGMA-milo/SystolicBP_leprinject_gwas_milo

In [ ]:
# Get proportions of cells per condition in a neighborhood
# Create a dummy matrix for groups
group_matrix <- model.matrix(~ 0 + factor(colData(milo_lepip)$group))
colnames(group_matrix) <- levels(factor(colData(milo_lepip)$group))
  
# Perform matrix multiplication
result <- t(group_matrix) %*% nh_matrix

# Calculate column sums
col_sums <- colSums(result)

# Calculate proportions
proportions <- sweep(result, 2, col_sums, FUN = "/")  %>% 
    data.frame()  %>% 
    rownames_to_column("group")  %>% 
    pivot_longer(-group)  %>% 
    mutate(name = gsub("X","", name))


In [ ]:
# read in magma results
gwas <- "SystolicBP_lepinject24"
res <- read.table(paste0("/projects/perslab/people/jmg776/ad_hoc/dylan_MAGMA-milo/", gwas, "_gwas_milo.gsa.out"), header=T) %>%  data.frame()  %>% 
    mutate(VARIABLE = gsub("X","", VARIABLE))  %>% 
    dplyr::rename(Nhood=VARIABLE)  %>% 
    dplyr::select(Nhood, BETA, P)   %>% 
    mutate(padj = p.adjust(P)) 

# res  %>% bind_cols(da_results_hfd)  %>% 
#     group_by(predicted.celltype)  %>% 
#     summarise(mean = max(-log10(P)))  %>% 
#     arrange(desc(mean))

# a <- res  %>% bind_cols(da_results_hfd)  %>% mutate(logFCwt = da_results_fast$logFC, nhood = as.character(`Nhood...10`))  %>% 
#     inner_join(proportions, by=c("nhood"="name"))  %>% 
#     filter(predicted.celltype%in%c("1")), predicted.celltype_fraction>0.7, value>0.1) %>% 
#     group_by(group.y)  %>% 
#     #summarise(cor = cor(-log10(P), value))
#     nest(-group.y)  %>%  
#     mutate(res = purrr::map(data, function(x) lm(-log10(P) ~ value, data=x)  %>% broom::tidy(.)))  %>%  unnest(res) %>%
#     select(-data)
# write_tsv(a, paste0("/projects/perslab/people/jmg776/ad_hoc/dylan_MAGMA-milo/", gwas, "_significance.tsv"))
res  %>% bind_cols(da_results_hfd)  %>% mutate(logFCwt = da_results_fast$logFC, nhood = as.character(`Nhood...10`))  %>% 
    inner_join(proportions, by=c("nhood"="name"))  %>% 
    filter(predicted.celltype%in%c("12"), predicted.celltype_fraction>0.7, value>0) %>% 
    separate(group.y, into = c("geno", "diet"))  %>% 
    ggplot() +
    aes(x=value, y= -log10(P))+ geom_point() +
    geom_smooth(method="lm") + 
    coord_cartesian(ylim = c(0, 4)) +
    facet_grid(vars(geno), vars(diet)) +
    cowplot::theme_half_open() + cowplot::background_grid() + cowplot::panel_border()

pdf("Lep-inj-24_GWAS-systolicBP.pdf", width = 10, height = 10)
res  %>% bind_cols(da_results_hfd)  %>% mutate(logFCwt = da_results_fast$logFC, nhood = as.character(`Nhood...10`))  %>% 
    inner_join(proportions, by=c("nhood"="name"))  %>% 
    filter(predicted.celltype%in%c("12"), predicted.celltype_fraction>0.7, value>0) %>% 
    separate(group.y, into = c("geno", "diet"))  %>% 
    ggplot() +
    aes(x=value, y= -log10(P))+ geom_point() +
    geom_smooth(method="lm") + 
    coord_cartesian(ylim = c(0, 4)) +
    facet_grid(vars(geno), vars(diet)) +
    cowplot::theme_half_open() + cowplot::background_grid() + cowplot::panel_border()
dev.off()